# Phase 6: Deployment & Integration Pipeline

## Executive Overview
This notebook serves as the operational **Inference & Batch Prediction Engine** for the Customer Churn Intelligence System. Key pipeline steps include:

1. **Production Artifact Ingestion:** Load the persisted champion model bundle containing the binary classifier, multi-horizon risk classifier, and feature schema.
2. **Batch Inference Execution:** Score unseen customer feature records to calculate binary churn probabilities, binary predictions, and multi-horizon risk tiers (**Active <3 Mo**, **Moderate Risk 3-6 Mo**, **High Risk 6-9 Mo**, **Dormant 9+ Mo**).
3. **Action Routing Engine:** Automatically assign targeted business retention actions based on risk tiers (e.g., immediate high-value discounts vs. automated win-back campaigns).
4. **Database Persistence:** Persist scored customer profiles into PostgreSQL (`customer_churn_predictions`) using atomic transactions for business dashboard consumption (Power BI / Grafana / Metabase).

### Step 1: Environment Setup & Strategic Path Resolution
We initialize system dependencies, verify workspace directory structures, and configure logging boundaries for batch inference execution.

In [3]:
# ==============================================================================
# STEP 1: ENVIRONMENT SETUP & PATH RESOLUTION
# ==============================================================================
import os
import sys
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# Path Resolution Framework
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'processed_predictions'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Point to model_v1.pkl inside the models directory
MODEL_BUNDLE_PATH = MODELS_DIR / 'model_v1.pkl'
FEATURES_PATH = DATA_DIR / 'customer_features_matrix.parquet'
OUTPUT_PREDICTIONS_PATH = REPORTS_DIR / 'scored_customer_churn_predictions.parquet'

print(f"[OK] Project Root:     {PROJECT_ROOT}")
print(f"[OK] Model Bundle:     {MODEL_BUNDLE_PATH}")
print(f"[OK] Features Source:  {FEATURES_PATH}")
print(f"[OK] Output Path:      {OUTPUT_PREDICTIONS_PATH}")

[OK] Project Root:     C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System
[OK] Model Bundle:     C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\models\model_v1.pkl
[OK] Features Source:  C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\data\processed\customer_features_matrix.parquet
[OK] Output Path:      C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\reports\processed_predictions\scored_customer_churn_predictions.parquet


### Step 2: Model Bundle & Artifact Loading
We load the serialized production bundle containing:
- Binary 90-Day Churn Pipeline
- Multi-Horizon Risk Tier Pipeline (3 / 6 / 9 Months)
- Enforced Feature Schema

In [7]:
# ==============================================================================
# STEP 2: ARTIFACT INGESTION & DEFENSIVE SCHEMA VERIFICATION
# ==============================================================================
if not MODEL_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Model bundle not found at: {MODEL_BUNDLE_PATH}")

with open(MODEL_BUNDLE_PATH, 'rb') as f:
    artifact_bundle = pickle.load(f)

# Handle both dict-bundled artifacts and direct Pipeline objects safely
if isinstance(artifact_bundle, dict):
    binary_model = artifact_bundle.get('binary_model', artifact_bundle)
    multi_tier_model = artifact_bundle.get('multi_tier_model', binary_model)
    REQUIRED_FEATURES = artifact_bundle.get('feature_columns', [
        'Frequency_Orders', 'Monetary_Spend', 'Total_Units', 
        'Unique_Products', 'Avg_Order_Value', 'Avg_Units_Per_Order'
    ])
    champion_name = artifact_bundle.get('champion_name', type(binary_model.named_steps['classifier']).__name__)
else:
    binary_model = artifact_bundle
    multi_tier_model = artifact_bundle
    REQUIRED_FEATURES = [
        'Frequency_Orders', 'Monetary_Spend', 'Total_Units', 
        'Unique_Products', 'Avg_Order_Value', 'Avg_Units_Per_Order'
    ]
    champion_name = type(binary_model.named_steps['classifier']).__name__

print(f"[OK] Champion Architecture Loaded: '{champion_name}'")
print(f"[OK] Required Features ({len(REQUIRED_FEATURES)}): {REQUIRED_FEATURES}")

FileNotFoundError: Model bundle not found at: C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\models\model_v1.pkl

### Step 3: Real-Time Scoring & Multi-Horizon Action Routing
We generate probability scores for binary churn and map multi-tier predictions to specific business retention actions:
- **`Active (<3 Mo)`:** No Intervention required / Loyalty perks.
- **`Moderate Risk (3-6 Mo)`:** Re-engagement survey & personalized product bundles.
- **`High Risk (6-9 Mo)`:** Urgent high-value retention offer / Direct outreach.
- **`Dormant (9+ Mo)`:** Automated win-back campaign / Clearance offers.

In [ ]:
# ==============================================================================
# STEP 3: BATCH INFERENCE & ACTION ROUTING ENGINE
# ==============================================================================
if not FEATURES_PATH.is_file():
    raise FileNotFoundError(f"Feature matrix missing at: {FEATURES_PATH}")

df_customers = pd.read_parquet(FEATURES_PATH)

# Verify schema
missing_cols = set(REQUIRED_FEATURES) - set(df_customers.columns)
if missing_cols:
    raise ValueError(f"Feature matrix is missing required predictors: {missing_cols}")

X_infer = df_customers[REQUIRED_FEATURES].copy()

# Generate Inference Scores
df_scored = df_customers[['Customer_ID']].copy()

# 1. Binary Predictions & Probabilities
df_scored['Churn_Probability'] = binary_model.predict_proba(X_infer)[:, 1].round(4)
df_scored['Is_Churned_Predicted'] = binary_model.predict(X_infer)

# 2. Multi-Horizon Risk Tier Prediction
risk_tier_map = {
    0: 'Active (<3 Mo)',
    1: 'Moderate Risk (3-6 Mo)',
    2: 'High Risk (6-9 Mo)',
    3: 'Dormant (9+ Mo)'
}

action_routing_map = {
    'Active (<3 Mo)': 'Maintain Standard Engagement & Loyalty Rewards',
    'Moderate Risk (3-6 Mo)': 'Send Re-engagement Survey & Tailored Recommendations',
    'High Risk (6-9 Mo)': 'Trigger Urgent Direct Discount Code (High-Value Retention)',
    'Dormant (9+ Mo)': 'Route to Automated Win-Back Email Campaign'
}

raw_tier_preds = multi_tier_model.predict(X_infer)
df_scored['Risk_Horizon_Tier'] = [risk_tier_map[pred] for pred in raw_tier_preds]
df_scored['Recommended_Business_Action'] = df_scored['Risk_Horizon_Tier'].map(action_routing_map)

print(f"[OK] Scored {len(df_scored):,} customer profiles successfully.")
print("\n=== SAMPLE SCORED CUSTOMER PREDICTIONS ===")
display(df_scored.head(10))

### Step 4: Atomic Local Export
We write the scored prediction dataset to disk in Parquet format using atomic write operations (`.tmp.parquet` $\rightarrow$ `.parquet`) to prevent file corruption during pipeline updates.

In [9]:
# ==============================================================================
# STEP 4: ATOMIC LOCAL STORAGE PERSISTENCE
# ==============================================================================
temp_output_path = OUTPUT_PREDICTIONS_PATH.with_suffix('.tmp.parquet')

df_scored.to_parquet(temp_output_path, index=False, engine='pyarrow')
temp_output_path.replace(OUTPUT_PREDICTIONS_PATH)

print(f"[OK] Scored customer predictions saved locally to:\n     {OUTPUT_PREDICTIONS_PATH.resolve()}")

NameError: name 'df_scored' is not defined

### Step 5: Production PostgreSQL Export
We persist the final scored predictions table (`customer_churn_predictions`) directly to PostgreSQL for consumption by BI dashboards and marketing automation tools.

In [5]:
# ==============================================================================
# STEP 5: LOCAL POSTGRESQL PRODUCTION SYNCHRONIZATION (VS CODE SAFE)
# ==============================================================================
import os
from sqlalchemy import create_engine, text

# Local PostgreSQL Credentials
DB_USER = "postgres"
DB_PASS = "2612"  # Your password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_churn"

# Try psycopg3 driver first, fallback to psycopg2 if needed
DATABASE_URL = f"postgresql+psycopg://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
FALLBACK_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

TABLE_NAME = "customer_churn_predictions"

print("Connecting to local PostgreSQL database...")

try:
    # Attempt Primary Connection (psycopg v3)
    try:
        engine = create_engine(DATABASE_URL, pool_pre_ping=True, connect_args={'connect_timeout': 5})
        conn_test = engine.connect()
        conn_test.close()
    except Exception:
        # Fallback to psycopg2 driver
        engine = create_engine(FALLBACK_URL, pool_pre_ping=True, connect_args={'connect_timeout': 5})

    # Atomic Write to Database
    with engine.begin() as connection:
        df_scored.to_sql(
            name=TABLE_NAME,
            con=connection,
            if_exists='replace',  # Replace table with latest predictions
            index=False,
            chunksize=5_000
        )
    
    # Verify Rows Inserted
    with engine.connect() as conn:
        db_count = conn.execute(text(f"SELECT COUNT(*) FROM {TABLE_NAME}")).scalar()
        
    print("=" * 70)
    print("POSTGRESQL SYNCHRONIZATION COMPLETED SUCCESSFULLY")
    print(f"Target Database: '{DB_NAME}'")
    print(f"Target Table:    '{TABLE_NAME}'")
    print(f"Total Rows Verified in Database: {db_count:,}")
    print("=" * 70)

except Exception as error:
    print(f"\n[ERROR] Connection failed: {type(error).__name__}: {error}")
    print("\nChecklist to fix:")
    print("1. Ensure PostgreSQL service is running on Windows.")
    print(f"2. Ensure Database '{DB_NAME}' is created in pgAdmin.")
    print("3. Verify password '2612' is correct for user 'postgres'.")
finally:
    if 'engine' in locals():
        engine.dispose()

Connecting to local PostgreSQL database...

[ERROR] Connection failed: ModuleNotFoundError: No module named 'psycopg2'

Checklist to fix:
1. Ensure PostgreSQL service is running on Windows.
2. Ensure Database 'customer_churn' is created in pgAdmin.
3. Verify password '2612' is correct for user 'postgres'.
